# 08 — Constraint Authoring

COGANT's **CONSTRAINT** role captures code that enforces invariants — things that must be true for the system to be well-formed. Python functions named `check_*`, `validate_*`, `assert_*`, plus classes named `*Checker` / `*Validator`, are the canonical triggers.

In this notebook we:
1. Write a small, self-contained Python module with explicit constraint functions.
2. Run the forward pipeline on it and inspect the CONSTRAINT mapping.
3. **Add a new constraint**, re-run, and compare the ε (epsilon) delta in the semantic mapping.

**Run from the `cogant/` directory:**
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/08_constraint_authoring.ipynb
```

## 1. A tiny constraint module

We write the module to a temp directory so the notebook is hermetic. The module contains:

- `ModelChecker` class with `validate_distribution`, `check_transition_matrix`, `assert_positive` methods.
- A top-level `check_dimensionality` helper.

Every symbol exercises a different CONSTRAINT trigger keyword.

In [1]:
from __future__ import annotations

import tempfile
import textwrap
from pathlib import Path

WORKDIR_V1 = Path(tempfile.mkdtemp(prefix="cogant_nb08_v1_"))
MODULE_V1 = WORKDIR_V1 / "constraints.py"

MODULE_V1.write_text(
    textwrap.dedent('''
    """Constraint module v1 — three built-in checks."""
    from __future__ import annotations


    class ModelChecker:
        """Validates structural invariants on small Active Inference models."""

        def __init__(self, tolerance: float = 1e-6) -> None:
            self.tolerance = tolerance
            self._violations: list[str] = []

        def validate_distribution(self, dist: list[float]) -> bool:
            """A distribution must sum to 1 and be non-negative."""
            self._violations.clear()
            if any(v < -self.tolerance for v in dist):
                self._violations.append("negative values")
            if abs(sum(dist) - 1.0) > self.tolerance:
                self._violations.append("sum != 1")
            return not self._violations

        def check_transition_matrix(self, matrix: list[list[float]]) -> bool:
            """Each column of a transition matrix must sum to 1."""
            self._violations.clear()
            if not matrix or not matrix[0]:
                return False
            for col in range(len(matrix[0])):
                total = sum(matrix[row][col] for row in range(len(matrix)))
                if abs(total - 1.0) > self.tolerance:
                    self._violations.append(f"col {col} sums to {total}")
            return not self._violations

        def assert_positive(self, values: list[float]) -> None:
            for i, v in enumerate(values):
                if v < 0:
                    raise ValueError(f"values[{i}] = {v} < 0")

        def get_violations(self) -> list[str]:
            return list(self._violations)


    def check_dimensionality(matrix, rows: int, cols: int) -> bool:
        """Verify matrix shape matches expectations."""
        if len(matrix) != rows:
            return False
        return all(len(row) == cols for row in matrix)
''').lstrip()
)

print(f"Wrote {MODULE_V1}")
print(MODULE_V1.read_text().splitlines()[0])

Wrote /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb08_v1_y86b3ogv/constraints.py
"""Constraint module v1 — three built-in checks."""


## 2. Forward pipeline — version 1

Point a `Session` at the v1 module, build the graph, and translate it to the GNN semantic mapping. We then count how many nodes landed in CONSTRAINT.

In [2]:
from collections import Counter

from cogant.api.session import Session


def _role_of(mapping) -> str:
    kind = getattr(mapping, "kind", None)
    if kind is None and isinstance(mapping, dict):
        kind = mapping.get("kind")
    return getattr(kind, "value", str(kind)).upper() if kind is not None else "UNKNOWN"


def analyse(repo: Path) -> dict:
    session = Session(repo_path=repo)
    graph = session.build_graph()
    session.translate_to_gnn()
    semantic_mappings = session._bundle.artifacts.get("_semantic_mappings", {})  # noqa: SLF001

    role_counts: Counter = Counter()
    constraint_nodes: list[str] = []
    for mapping in semantic_mappings.values():
        role = _role_of(mapping)
        role_counts[role] += 1
        if role != "CONSTRAINT":
            continue
        fragment_ids = getattr(mapping, "graph_fragment_node_ids", None) or []
        for nid in fragment_ids:
            node = graph["nodes"].get(nid, {})
            name = node.get("qualified_name") or node.get("name") or nid
            constraint_nodes.append(name)
        if not fragment_ids:
            label = getattr(mapping, "semantic_label", None) or "<unlabeled>"
            constraint_nodes.append(label)
    return {
        "node_count": len(graph.get("nodes", {})),
        "edge_count": len(graph.get("edges", {})),
        "role_counts": role_counts,
        "constraint_nodes": sorted(set(constraint_nodes)),
        "total_mapped": sum(role_counts.values()),
    }


result_v1 = analyse(WORKDIR_V1)
print(f"v1 nodes      : {result_v1['node_count']}")
print(f"v1 edges      : {result_v1['edge_count']}")
print(f"v1 mapped     : {result_v1['total_mapped']}")
print(f"v1 CONSTRAINT : {result_v1['role_counts'].get('CONSTRAINT', 0)}")
print("v1 constraint nodes:")
for name in result_v1["constraint_nodes"]:
    print(f"    - {name}")

v1 nodes      : 8
v1 edges      : 12
v1 mapped     : 7
v1 CONSTRAINT : 4
v1 constraint nodes:
    - constraints.ModelChecker.assert_positive
    - constraints.ModelChecker.check_transition_matrix
    - constraints.ModelChecker.validate_distribution
    - constraints.check_dimensionality


## 3. Add a new constraint and re-run

Version 2 of the module adds a single new function — `validate_preference_vector` — that enforces the C vector is finite and bounded. We write it to a *fresh* temp directory so the pipeline can't accidentally reuse a prior graph cache, then re-run `analyse(...)`.

In [3]:
WORKDIR_V2 = Path(tempfile.mkdtemp(prefix="cogant_nb08_v2_"))
MODULE_V2 = WORKDIR_V2 / "constraints.py"

v1_text = MODULE_V1.read_text()
new_fn = textwrap.dedent('''
    def validate_preference_vector(c: list[float], *, max_magnitude: float = 100.0) -> bool:
        """Preference vector C must be finite and bounded in magnitude."""
        if not c:
            return False
        for v in c:
            if v != v:  # NaN
                return False
            if abs(v) > max_magnitude:
                return False
        return True
''')
MODULE_V2.write_text(v1_text + new_fn)
print(f"Wrote {MODULE_V2}")

result_v2 = analyse(WORKDIR_V2)
print(f"v2 nodes      : {result_v2['node_count']}")
print(f"v2 mapped     : {result_v2['total_mapped']}")
print(f"v2 CONSTRAINT : {result_v2['role_counts'].get('CONSTRAINT', 0)}")
print("v2 constraint nodes:")
for name in result_v2["constraint_nodes"]:
    print(f"    - {name}")

Wrote /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb08_v2_05t7_jl5/constraints.py
v2 nodes      : 9
v2 mapped     : 8
v2 CONSTRAINT : 5
v2 constraint nodes:
    - constraints.ModelChecker.assert_positive
    - constraints.ModelChecker.check_transition_matrix
    - constraints.ModelChecker.validate_distribution
    - constraints.check_dimensionality
    - constraints.validate_preference_vector


## 4. The ε delta

We define ε as the relative change in CONSTRAINT coverage: how much of the graph's mapped surface area now belongs to the CONSTRAINT role after adding one function? This is a useful signal when iterating on a DSL ruleset or a custom plugin — small deltas mean the new rule is conservative; large deltas usually mean a glob pattern is too loose.

In [4]:
def _constraint_fraction(result: dict) -> float:
    total = result["total_mapped"] or 1
    return result["role_counts"].get("CONSTRAINT", 0) / total


frac_v1 = _constraint_fraction(result_v1)
frac_v2 = _constraint_fraction(result_v2)
epsilon = frac_v2 - frac_v1

print(f"CONSTRAINT fraction v1: {frac_v1:.3f}")
print(f"CONSTRAINT fraction v2: {frac_v2:.3f}")
print(f"\u03b5 (delta)          : {epsilon:+.3f}")
print()
added = sorted(set(result_v2["constraint_nodes"]) - set(result_v1["constraint_nodes"]))
print("Newly CONSTRAINT-mapped nodes:")
for name in added:
    print(f"    + {name}")
if not added:
    print("    (none — the new function was not picked up)")

CONSTRAINT fraction v1: 0.571
CONSTRAINT fraction v2: 0.625
ε (delta)          : +0.054

Newly CONSTRAINT-mapped nodes:
    + constraints.validate_preference_vector


## 5. What to do when ε is zero

If `validate_preference_vector` **didn't** land in CONSTRAINT, the most likely reason is that the built-in rule set's `validate_*` trigger matched only *methods*, not free functions. That's exactly the kind of gap you patch with:

- **a YAML DSL rule** (notebook 10) — three lines of YAML declaring `name_pattern: 'validate_*'` with `node_kind: function`, or
- **a custom plugin** (notebook 09) — if you need logic the DSL can't express.

Either way you re-run `analyse(...)` and re-measure ε to confirm the fix.

## 6. Takeaways

- CONSTRAINT authoring is as simple as writing functions with the right name prefix.
- You can measure the impact of a new constraint with a single forward-pipeline delta — no need to inspect the GNN markdown by hand.
- An ε of zero after adding a rule usually means the built-in trigger is too narrow; extend it via DSL (notebook 10) or a plugin (notebook 09) and re-measure.